# 03 - Evaluation & Threshold Tuning
## Menghitung F1, Precision, Recall, AUC-ROC per Model

In [ ]:
!pip install -q scikit-learn matplotlib pandas numpy

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, roc_curve, confusion_matrix

with open('data/similarities.pkl', 'rb') as f:
    results = pickle.load(f)

y_true = results['y_true']
sims = {
    'TF-IDF': results['sims_tfidf'],
    'SBERT': results['sims_sbert'],
    'IndoBERT': results['sims_indobert']
}

print(f"Loaded {len(y_true)} test pairs")

## Threshold Tuning

In [ ]:
def find_optimal_threshold(y_true, y_scores):
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    optimal_idx = np.argmax(tpr - fpr)
    return thresholds[optimal_idx]

thresholds = {}
for name, scores in sims.items():
    thresh = find_optimal_threshold(y_true, scores)
    thresholds[name] = thresh
    print(f"{name} - Optimal threshold: {thresh:.4f}")

## Hitung Metrik

In [ ]:
metrics_list = []

for name, scores in sims.items():
    thresh = thresholds[name]
    y_pred = (scores >= thresh).astype(int)
    
    metrics = {
        'model': name,
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'auc': roc_auc_score(y_true, scores),
        'threshold': thresh
    }
    metrics_list.append(metrics)
    
    print(f"\n=== {name} ===")
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")
        else:
            print(f"  {k}: {v}")

df_metrics = pd.DataFrame(metrics_list)
print("\n=== SUMMARY TABLE ===")
print(df_metrics.to_string(index=False))

## ROC Curve

In [ ]:
os.makedirs('assets', exist_ok=True)

plt.figure(figsize=(10, 8))
for name, scores in sims.items():
    fpr, tpr, _ = roc_curve(y_true, scores)
    auc = roc_auc_score(y_true, scores)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Perbandingan Model')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig('assets/roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## Confusion Matrix

In [ ]:
for name, scores in sims.items():
    thresh = thresholds[name]
    y_pred = (scores >= thresh).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(6, 5))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title(f'Confusion Matrix - {name}')
    plt.colorbar()
    plt.xticks([0, 1], ['Negative', 'Positive'])
    plt.yticks([0, 1], ['Negative', 'Positive'])
    for i in range(2):
        for j in range(2):
            plt.text(j, i, str(cm[i, j]), ha='center', va='center', color='red', fontsize=14)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f'assets/confusion_matrix_{name.lower()}.png', dpi=150, bbox_inches='tight')
    plt.show()

## Simpan Hasil Evaluasi

In [ ]:
df_metrics.to_csv('assets/hasil_evaluasi.csv', index=False)
print("Saved: assets/hasil_evaluasi.csv")
print("\nFile di assets/:")
for f in os.listdir('assets'):
    print(f"  - assets/{f}")